<a href="https://colab.research.google.com/github/Joy-Kimani/slm-arm64-benchmark/blob/main/04_quality_eval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Measure the quality of each GGUF version

This notebook runs the f16, Q8_0 and Q4_0 files over the same TinyStories validation text and converts the scores into bits per byte(BPB).

**Prerequisites**: Run "03_export_to_gguf" first.

## Step 1: Mount Google Drive and set the paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import math
import json
import subprocess
import re

out_dir = '/content/drive/MyDrive/slm70m'

Mounted at /content/drive


In [ ]:
!pip install -q datasets tokenizers

## Step 2: Write the validation text

We extract the first 1,000 stories from the TinyStories validation set and save them to a single text file separated by blank lines. We also count the total tokens and UTF-8 bytes so we can calculate the average bytes per token later when converting perplexity into bits per byte.

In [ ]:
from datasets import load_dataset
from tokenizers import Tokenizer

ds = load_dataset("roneneldan/TinyStories", split="validation")
tokenizer = Tokenizer.from_file(os.path.join(out_dir, 'tokenizer.json'))

texts = ds[:1000]['text']
val_text = "\n\n".join(texts)
open('/content/validation.txt', 'w').write(val_text)

n_tokens = sum(len(e.ids) for e in tokenizer.encode_batch(texts))
n_bytes = len(val_text.encode('utf-8'))
print(n_tokens, n_bytes, n_bytes / n_tokens)
print("chunks of 256 tokens:", n_tokens // 256)

README.md:   0%|          | 0.00/1.06k [00:00<?, ?B/s]

data/train-00000-of-00004-2d5a1467fff108(…): reconstructing file:   0%|          |  0.00B /  249MB            

data/train-00000-of-00004-2d5a1467fff108(…): downloading bytes:           |  0.00B            

data/train-00001-of-00004-5852b56a2bd28f(…): reconstructing file:   0%|          |  0.00B /  248MB            

data/train-00001-of-00004-5852b56a2bd28f(…): downloading bytes:           |  0.00B            

data/train-00002-of-00004-a26307300439e9(…): reconstructing file:   0%|          |  0.00B /  246MB            

... (4 outputs left out)

data/validation-00000-of-00001-869c898b5(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/2119719 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/21990 [00:00<?, ? examples/s]

188759 794373 4.208398010161105
chunks of 256 tokens: 737


## Step 3: Build the perplexity tool from llama.cpp

We use the same llama.cpp version as in the export notebook, keeping the tooling consistent and then compile the llama-perplexity evaluation binary. Limit the build to 2 parallel jobs(-j 2) so Colab does not run out of RAM during complilation.


In [ ]:
!git clone https://github.com/ggml-org/llama.cpp || true
!git -C llama.cpp checkout f8caa5ea30b5775512906ae3b7ee53b21a7a6e1d

Cloning into 'llama.cpp'...
remote: Enumerating objects: 167445, done.
remote: Counting objects: 100% (172/172), done.
remote: Compressing objects: 100% (93/93), done.
remote: Total 167445 (delta 94), reused 83 (delta 79), pack-reused 167273 (from 3)
Receiving objects: 100% (167445/167445), 492.96 MiB | 29.83 MiB/s, done.
Resolving deltas: 100% (120064/120064), done.
Updating files: 100% (4609/4609), done.
Note: switching to 'f8caa5ea30b5775512906ae3b7ee53b21a7a6e1d'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false

HEAD is no

In [ ]:
!cmake -S llama.cpp -B llama.cpp/build -DLLAMA_CURL=OFF -DCMAKE_BUILD_TYPE=Release
!cmake --build llama.cpp/build -j 2 --target llama-perplexity

-- The C compiler identification is GNU 13.3.0
-- The CXX compiler identification is GNU 13.3.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- llama.cpp version: 1.0.0-dev
CMAKE_BUILD_TYPE=Release
-- Found Git: /usr/bin/git (found version "2.43.0")
-- The ASM compiler identification is GNU
-- Found assembler: /usr/bin/cc
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD - Success
-- Found Threads: TRUE
-- Warning: ccache not found - consider installing it for faster compilation or disable this warning with GGML_CCACHE=OFF
-- CMAKE_SYSTEM_PROCESSOR: x86_64
-- GGML_SYSTEM_ARCH: x86
-- Found Ope

## Step 4: Measure the perplexity for each version

llama-perplexity cuts the text into chunks of 256 tokens, the context length the model was trained with. It only scores the second half of each chunk, so every scored token has at least 128 tokens of prior context. That means these numbers will not match the validation loss from training,treat these numbers as a relative comparison across GGUF versions rather than a direct match to training logs.

In [ ]:
n_chunks = 400
results = {}

for name in ['f16', 'q8_0', 'q4_0']:
    cmd = ['llama.cpp/build/bin/llama-perplexity',
           '-m', f'{out_dir}/slm70m-{name}.gguf',
           '-f', '/content/validation.txt',
           '-c', '256', '--chunks', str(n_chunks),
           '-t', str(os.cpu_count())]
    r = subprocess.run(cmd, capture_output=True, text=True)
    m = re.search(r'Final estimate: PPL = ([\d.]+) \+/- ([\d.]+)', r.stdout + r.stderr)
    results[name] = (float(m.group(1)), float(m.group(2)))
    print(name, results[name])

f16 (3.5561, 0.03013)
q8_0 (3.5565, 0.03013)
q4_0 (3.5853, 0.03043)


## Step 5: Convert to bits per byte and save the results

Bits per byte = log2(perplexity) / bytes per token. The bytes per token comes from the whole validation text, so it is an approximation of the scored half.
Finally, we print a summary table showing the trade-off vs. f16 and save the results to quality.json.

In [ ]:
bytes_per_token = n_bytes / n_tokens
base_bpb = None
table = {}

for name, (ppl, se) in results.items():
    bpb = math.log2(ppl) / bytes_per_token
    size_mb = os.path.getsize(f'{out_dir}/slm70m-{name}.gguf') / 1e6
    if base_bpb is None:
        base_bpb = bpb
    table[name] = {'size_mb': round(size_mb, 1), 'ppl': ppl, 'bits_per_byte': bpb}
    print(f"{name:5s}  size {size_mb:6.1f} MB   perplexity {ppl:.4f}   bits/byte {bpb:.4f}   change vs f16 {bpb - base_bpb:+.4f}")

json.dump(table, open(os.path.join(out_dir, 'quality.json'), 'w'), indent=2)

f16    size  140.1 MB   perplexity 3.5561   bits/byte 0.4349   change vs f16 +0.0000
q8_0   size   74.7 MB   perplexity 3.5565   bits/byte 0.4350   change vs f16 +0.0000
q4_0   size   45.0 MB   perplexity 3.5853   bits/byte 0.4377   change vs f16 +0.0028
